In [14]:
from SymbolicDSGE import load_bundle
from SymbolicDSGE.estimation.spec import (
    MCMCResultMeta,
    OptimizationResultMeta,
)
from SymbolicDSGE.estimation.results import MCMCResult
import numpy as np

In [2]:
loaded = load_bundle("experiment-1.sdsge")

In [3]:
print("Created by:", loaded.manifest.created_by)
print("Created at:", loaded.manifest.created_at)
print("Format version:", loaded.manifest.sdsge_version)

Created by: experiment-1
Created at: 2026-06-13T10:29:08.789883+00:00
Format version: 1


In [4]:
reference = loaded.reference
if reference is not None:
    print("Stable:", reference.policy.stab == 0)
    print("Eigenvalues:", reference.policy.eig)
    print("A shape:", reference.A.shape)

Stable: True
Eigenvalues: [0.28018451+0.j 0.83      +0.j 0.85      +0.j 2.60451546+0.j
 1.18546572+0.j]
A shape: (5, 5)


In [6]:
T = 20
rng = np.random.default_rng(42)
shocks = {
    "g,z": rng.standard_normal((T, 2)),
}
sim = reference.sim(
    T=T,
    shocks=shocks,
    observables=True,
)
print(sim["Infl"][:5])

[  3.43         8.20387037  10.83531974 -12.17755328   0.91492503]


In [7]:
estimation = loaded.estimation

if estimation is not None:
    print("Method:", estimation.spec.method)
    print("Observables:", estimation.spec.observables)
    print("Parameters:", [p.name for p in estimation.spec.parameters])

Method: map
Observables: ['Infl', 'Rate']
Parameters: ['psi_pi', 'psi_x']


In [9]:
result = estimation.result
if isinstance(result, OptimizationResultMeta):
    print("Point estimate:", result.theta)
    print("Log-posterior:", result.logpost)
elif isinstance(result, MCMCResultMeta):
    print("Acceptance:", result.accept_rate)
    print("Draws:", result.n_draws, "burn-in:", result.burn_in)

Point estimate: {'psi_pi': 1.48, 'psi_x': 0.55}
Log-posterior: -87.3


In [10]:
if estimation.observed is not None:
    print("Observed shape:", estimation.observed.shape)
if estimation.posterior is not None:
    samples = estimation.posterior["samples"]
    print("Posterior mean:", samples.mean(axis=0))

Observed shape: (40, 2)


In [18]:
mc = loaded.mc

if mc is not None:
    print("Pipeline nodes:", [n.id for n in mc.spec.nodes])
    print("Pipeline edges:", [(e.source, e.target) for e in mc.spec.edges])

    if mc.document is not None:
        wire = mc.wire()
        print("Run kind:", wire["kind"])

Pipeline nodes: ['sim', 'jb']
Pipeline edges: [('sim', 'jb')]


In [19]:
sim_spec = loaded.simulation

if sim_spec is not None:
    print("T:", sim_spec.T)
    print("Observables flag:", sim_spec.observables)
    if sim_spec.shock_generation is not None:
        print("Seed:", sim_spec.shock_generation.seed)
        print("Distribution:", sim_spec.shock_generation.dist)

T: 25
Observables flag: True
Seed: 42
Distribution: norm
